# Dropout en Pytorch con clase y con Función

In [1]:
# Importar librerías
import torch
import torch.nn as nn
import torch.nn.functional as F

Imaginemos que queremos entrenar un modelo de Deep Learning: la idea es que durante el entrenamiento, en cada iteración o época, **algunas unidades del modelo (TLU) se eliminen de forma aleatoria**.

Cuando decimos eliminarlas, nos referimos a que **sus activaciones son forzadas a cero**. Sin importar los valores de los pesos o de las entradas, la salida de estas unidades se desconecta temporalmente. En la siguiente iteración se seleccionan otras activaciones distintas al azar; la clave del **dropout** es asignar una probabilidad fija de caída ($p$), de modo que en cada paso cada unidad tiene una probabilidad dada de ser descartada.

### ¿Por qué se utiliza?

* **Representaciones distribuidas:** Obliga a la red a no depender de unas pocas neuronas específicas, sino a aprender características distribuidas por todas las TLU.
* **Robustez y estabilidad:** Hace que el modelo sea mucho más resistente al ruido y a las fluctuaciones en los datos.
* **Mejor generalización:** Evita el sobreajuste (*overfitting*), especialmente cuando los datos de entrenamiento son muy similares entre sí.

### El inconveniente técnico

Por otro lado, al apagar aleatoriamente una fracción de las neuronas, **la suma total de las activaciones se reduce** debido a la gran cantidad de ceros introducidos. Esto puede traer consecuencias negativas para los cálculos posteriores de las capas siguientes, incluido el momento de pasar por la función **softmax** (a menos que se compense escalando adecuadamente las activaciones durante el entrenamiento, como ocurre en la implementación estándar de PyTorch).

---

existen **dos soluciones matemáticas equivalentes**, aunque en la práctica los frameworks modernos (como PyTorch) utilizan la primera:

1. **Escalar durante el entrenamiento (Inverted Dropout / Escalado en entrenamiento):**
Consiste en dividir las activaciones supervivientes entre $(1 - p)$ (donde $p$ es la probabilidad de dropout) justo en la capa de entrenamiento. De esta forma, la magnitud global se mantiene intacta y **no hay que tocar nada cuando el modelo pasa a modo evaluación (`eval()`)**, ya que el modelo se queda tal cual.

2. **Reducir los pesos durante la evaluación (Escalado en prueba):**
Consiste en entrenar de forma estándar con el dropout "puro" (sin escalar en las capas) y luego, al terminar de entrenar y pasar a producción o evaluación, **multiplicar todos los pesos del modelo por $(1 - p)$** para compensar que durante el entrenamiento las neuronas recibían señales más débiles o menos frecuentes.

PyTorch y la gran mayoría de librerías modernas implementan la primera opción (*Inverted Dropout*): escalan los valores al vuelo durante el entrenamiento para que, al llegar a la fase de inferencia o producción, la red funcione directamente sin necesidad de modificar pesos ni aplicar factores correctivos.

In [2]:
# Empiricamente funciona para ciertos modelos, ciertos conjuntos de datos, esto hace que la neurona no cargue con demasiada responsabilidad

Esto hace que el modelo sea mas robusto y mas estable, el dropout suele ser mas estable tanto para LLMS como para DL como modelos de computer vision o etc

### Usando Dropout

In [3]:
# Definir una instancia de dropout y crear algunos datos
prob = .2

dropout = nn.Dropout(p=prob) #clase de nn.Dropout lo que nos importa es el parametro prob
x = torch.ones(100)
print('Vector de datos inicial:\n  ', x, '\n')

# Veamos qué devuelve dropout
y = dropout(x)
print('Vector post-dropout "en bruto":\n  ', y, '\n')

# Normalizado
print('Vector post-dropout normalizado:\n  ', y * (1 - prob), '\n')

# ¿Cuántos fueron eliminados?
print(f'{100*torch.sum(y==0)/len(x):.1f}% eliminados (esperado {prob*100:.1f}%)') # Normalizamos para ver facil lo que arroja

Vector de datos inicial:
   tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]) 

Vector post-dropout "en bruto":
   tensor([1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500,
        0.0000, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000,
        1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500, 0.0000, 0.0000,
        1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
      

In [4]:
x.sum(), y.sum(), (1-prob)*y.sum()

(tensor(100.), tensor(98.7500), tensor(79.))

Como se trata de una distribucion probabilistica no sera exactamente 20% si no alrededor de la media en torno al 20%

### Usando Funciones

In [5]:
F.dropout(x,p=prob)

tensor([1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 0.0000, 1.2500,
        0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500])

## El boton del dropout

In [10]:
# El dropout se desactiva al evaluar el modelo
dropout.eval() # tipicamente seria model.eval() si definimos un modelo
y = dropout(x)
print(y)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [15]:
# F.dropout() no se desactiva en modo eval:
dropout.eval()
y = F.dropout(x) #,p=prob (.5 es el valor por defecto)
print(y)

tensor([2., 0., 0., 2., 2., 0., 0., 2., 0., 0., 2., 2., 0., 2., 2., 2., 0., 0.,
        0., 2., 2., 2., 0., 2., 0., 0., 2., 0., 0., 2., 0., 0., 2., 0., 0., 2.,
        2., 0., 2., 0., 2., 2., 0., 0., 0., 2., 2., 2., 0., 2., 0., 0., 0., 0.,
        0., 0., 2., 2., 0., 0., 0., 0., 0., 2., 2., 0., 2., 0., 2., 2., 0., 0.,
        0., 0., 0., 0., 2., 0., 2., 2., 2., 2., 2., 2., 2., 0., 0., 2., 0., 0.,
        0., 2., 0., 0., 2., 2., 0., 0., 2., 2.])


In [12]:
# cuando el modelo esta en model.eval() es algo que se aplica durante el training no en la evualación, ni el despliegue o la inferencia

In [8]:
# Pero puedes desactivarlo manualmente
# dropout.eval() # Sin efecto :o
y = F.dropout(x, training=False)
print(y)

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


Se puede tener una funcion con una variable por separado

In [18]:
amitraining = True # false
y = F.dropout(x, training=amitraining)
print(y)

tensor([0., 2., 0., 2., 0., 0., 0., 2., 2., 2., 2., 2., 2., 2., 2., 0., 2., 0.,
        0., 0., 2., 0., 2., 2., 0., 0., 0., 0., 0., 0., 0., 2., 0., 0., 0., 2.,
        2., 2., 2., 2., 0., 2., 2., 0., 2., 2., 2., 0., 0., 2., 2., 0., 0., 0.,
        0., 2., 2., 2., 0., 2., 2., 2., 0., 0., 0., 2., 2., 2., 0., 0., 0., 0.,
        0., 0., 2., 0., 0., 2., 0., 0., 0., 2., 2., 0., 2., 2., 0., 0., 2., 0.,
        0., 2., 0., 0., 2., 0., 0., 0., 2., 2.])


In [9]:
# El modelo necesita ser restablecido tras cambiar al modo eval

dropout.train()
y = dropout(x)
print('Modo train() activado:\n', y, '\n') # Con dropout

dropout.eval()
y = dropout(x)
print('Modo eval() activado:\n', y, '\n') # Sin dropout

dropout.train()
y = dropout(x)
print('Modo train() reactivado:\n', y) # Dropout activo nuevamente ;)

Modo train() activado:
 tensor([1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 0.0000, 0.0000, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500,
        1.2500, 0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000,
        1.2500, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
        1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500,
        1.2500, 1.2500, 0.0000, 1.2500, 0.0000, 1.2500, 1.2500, 0.0000, 1.2500,
        1.2500, 0.0000, 0.0000, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
        0.0000, 1.2500, 1.2500, 1.2500, 1.2500, 0.0000, 1.2500, 1.2500, 1.2500,
        1.2500]) 

Modo eval() activado:
 tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.

En conclusión el dropout es una tecnica de regularización que suaviza las representaciones de la red apra evitar el modelo se ajuste en exceso a los datos para prevenir el overfitting del durante el entranamiento del modelo. LLMS usan menos dropout que otros modelos pero tambien se usa en algunas ocasiones